# Numerical primal semidefinite program solver for the two-box problem

## Context

In this notebook the optimal expected correct-guessing probability $\text{P}_\text{corr}^*$ for UD of the 2BP is numerically computed by using the *picos* package to solve complex semidefinite program (SDP). Specifically, we solve the following (primal) SDP:

$$
\begin{align}
    \text{P}_\text{corr}^* = \max_{M_A ,M_B}\quad & \eta_A\text{Tr}(M_A\rho_A) + \eta_B\text{Tr}(M_B\rho_B) & &\\
    \text{subject to } \quad & M_j \succcurlyeq 0 & j=A,B \\
    & \mathrm{I} - M_A - M_B \succcurlyeq 0 \\
    & M_A \rho_B = 0 \\
    & M_B \rho_A = 0. \\
\end{align}
$$

When *unbiased=True*, another constraint is added, namely

$$
\text{Tr}(M_A\rho_A)=\text{Tr}(M_B\rho_B).
$$

## Outputs

- Numerically computed minimum expected failure probability

In [105]:
import numpy as np
import picos

In [106]:
# simulation parameters


d = 4 # hilbert space dimension (2 qubits)

etaA = 1/3. # prior probability of Family A
etaB = 1 - etaA # prior probability of Family B
symmetric = False 
unbiased = False

In [107]:

import numpy as np

# 1. Definir las funciones de estado usando NUMPY estándar
def get_rho1(a):
    # |psi> = [sqrt(1-a^2), 0, a, 0]^T
    # Aseguramos que el término dentro de la raíz no sea negativo
    term = max(0, 1 - a**2)
    psi1 = np.array([np.sqrt(term), 0.0, a, 0.0], dtype=complex)
    rho = np.outer(psi1, np.conj(psi1))
    return rho

def get_rho2(b):
    # Family 2 mixture
    term = max(0, 1 - b**2)
    psi2 = np.array([0.0, np.sqrt(term), b, 0.0], dtype=complex)
    psi3 = np.array([0.0, -np.sqrt(term), b, 0.0], dtype=complex)
    rho = 0.5 * np.outer(psi2, np.conj(psi2)) + 0.5 * np.outer(psi3, np.conj(psi3))
    return rho

# 2. Configuración de la simulación
d = 4  # Dimensión fija para 2 qubits (tus estados son vectores de longitud 4)
n_samples = 5000 # Cantidad de muestras para aproximar la integral de la distribución

# 3. Generar distribuciones para a y b
# Distribución Uniforme entre 0 y 1
a_values = np.random.uniform(0.0, 1.0, n_samples)
# No es necesario clip si generamos directamente en el rango, pero lo dejamos por seguridad
a_values = np.clip(a_values, -1.0, 1.0) 

b_values = np.random.uniform(0.0, 1.0, n_samples)
b_values = np.clip(b_values, -1.0, 1.0)

# 4. Calcular las matrices de densidad promedio (Rho A y Rho B)
# Esto representa el estado "promedio" de cada familia
rhoA_samples = [get_rho1(a) for a in a_values]
rhoA = np.mean(rhoA_samples, axis=0)

rhoB_samples = [get_rho2(b) for b in b_values]
rhoB = np.mean(rhoB_samples, axis=0)

# Semidefinite Programming Solver

In [108]:
# set picos constants

rA = picos.Constant(etaA*rhoA)
rB = picos.Constant(etaB*rhoB)
I = picos.Constant(np.eye(d))

In [109]:
# sdp statement

P = picos.Problem()
MA = picos.HermitianVariable("MA", rA.shape)
MB = picos.HermitianVariable("MB", rA.shape)
P.set_objective("max", picos.trace(MA*rA + MB*rB))
for M in [MA,MB]: P.add_constraint( M >> 0)
P.add_constraint(I - MA - MB >> 0)
P.add_constraint(MA*rB == 0)
P.add_constraint(MB*rA == 0)

if unbiased:
    # Unbiased condition: Tr(MA * rhoA) == Tr(MB * rhoB)
    # Since rA = etaA * rhoA, we divide by etaA to cancel the prior
    P.add_constraint( (1/etaA) * picos.trace(MA*rA) == (1/etaB) * picos.trace(MB*rB) )

print(P)

Complex Semidefinite Program
  maximize tr(MA·[4×4] + MB·[4×4])
  over
    4×4 hermitian variables MA, MB
  subject to
    MA ≽ 0
    MB ≽ 0
    [4×4] - MA - MB ≽ 0
    MA·[4×4] = 0
    MB·[4×4] = 0


In [110]:
# solve

P.solve(solver = "cvxopt") #Encuentra las matrices MA y MB optimas
Perr = 1-P.value

In [111]:
# print and save data
fname_surfix = ""
if unbiased:
    fname_surfix += "unb_"

print("\nOptimal expected failure probability:", Perr)
print("\nOptimal expected Success probability:", 1 - Perr)
#print("Optimal X:", X.value, sep="\n")

# Ensure data directory exists
import os
if not os.path.exists("data"):
    os.makedirs("data")

# save optimal error probability
np.savetxt("data/"+fname_surfix+"perr_ud_primal2bp_num_N{:d}.txt".format(d), [Perr])

# save optimal measurement
# Use .value to get the numerical value from PICOS variables/constants
MAval = np.matrix(MA.value)
MBval = np.matrix(MB.value)
M0val = np.matrix(I.value) - MAval - MBval

np.save("data/"+fname_surfix+"MA_ud_2bp_num_N{:d}.npy".format(d), MAval)
np.save("data/"+fname_surfix+"MB_ud_2bp_num_N{:d}.npy".format(d), MBval)
np.save("data/"+fname_surfix+"M0_ud_2bp_num_N{:d}.npy".format(d), M0val)


Optimal expected failure probability: 0.33600558496119926

Optimal expected Success probability: 0.6639944150388007
